
# Using the HoloGradPy hardware interface

HoloGradPy uses a small native interface for cameras and SLMs.
This script walks through using the interface on simulated devices and provides an
example with real hardware at the end.

There are three entry points, all re-exported from ``hologradpy.hardware``.

- :func:`~hologradpy.hardware.factory.open_camera` and
  :func:`~hologradpy.hardware.factory.open_slm` construct a device and return it ready
  to use, in one call. This is the recommended way to obtain a device.

- :func:`~hologradpy.hardware.as_native.as_camera` and
  :func:`~hologradpy.hardware.as_native.as_slm` adapt a device you have already
  constructed. They are safe to call on a device that is already native.

- :func:`~hologradpy.hardware.factory.register_camera_backend` and
  :func:`~hologradpy.hardware.factory.register_slm_backend` give a driver class a short
  name, so ``open_camera`` and ``open_slm`` can build it by name, for example
  ``open_camera("thorcam", serial=...)``.

.. important::
   Geometry is ``(y, x) = (height, width)``. A per-axis quantity such as ``pixel_size``
   is ordered ``(y, x)``, and regions of interest are ``(row, col)``.

   Units are SI. ``pixel_size`` and ``wavelength`` are in metres, ``exposure`` is in
   seconds.


In [ ]:
import matplotlib.pyplot as plt
import torch

from hologradpy.hardware import (
    Camera,
    SLM,
    ROI,
    SimulatedSLMTorch,
    SimulatedCameraTorch,
    open_camera,
    open_slm,
    as_camera,
    as_slm,
    register_slm_backend,
)

from hologradpy.optics.complex_amplitude import ComplexAmplitude, FieldGeometry
from hologradpy.optics.systems import SLMFFTAffine
from hologradpy.optics.modules.slm_fields import PixelwiseSLMField
from hologradpy.profiles.amplitude import gaussian_beam_intensity
from hologradpy.utils import get_device
from hologradpy.visualizer import image_grid

device = get_device(verbose=True)

## Opening a device

For the purpose of this example, we use simulated hardware. Most code below is setting
up the optical model needed for simulated hardware. On real hardware you would go
straight to ``open_camera(YourDriver, ...)`` without any of this.



In [ ]:
slm_geometry = FieldGeometry(
    resolution=(1024, 1280),
    pixel_size=torch.tensor([12.5e-6, 12.5e-6], device=device),
    wavelength=torch.tensor(0.630e-6, device=device),
)

slm = open_slm(SimulatedSLMTorch, input_geometry=slm_geometry, bitdepth=8)

gaussian_intensity = gaussian_beam_intensity(
    *slm.get_spatial_grid(device), beam_radius=5e-3
)
beam = ComplexAmplitude(
    gaussian_intensity.sqrt() + 0j,
    wavelength=slm_geometry.wavelength,
    pixel_size=slm_geometry.pixel_size,
    power=1e-3,
)

camera_model = SLMFFTAffine(
    input_geometry=slm_geometry,
    virtual_slm=slm.virtual_slm,
    camera_resolution=(960, 1440),
    camera_pixel_size=(3.75e-6, 3.75e-6),
    focal_length=0.25,
    slm_field=PixelwiseSLMField(beam),
    padded_resolution=(2048, 2048),
    camera_angle=0,
    camera_shift=(0, 0),
    power_normalized=True,
)

camera = open_camera(
    SimulatedCameraTorch,
    slm_camera_model=camera_model,
    nd_filter_optical_density=5.0,
    quantum_efficiency=0.01,
)

## Reading geometry and units through the interface

These properties read the same way for a simulated device and for real hardware,
always in (y, x) order and SI units.



In [ ]:
print("SLM")
print("resolution (h, w):", slm.resolution)
print("pixel_size (y, x):", slm.pixel_size, "m")
print("wavelength:", slm.wavelength, "m")

print("Camera")
print("resolution (h, w):", camera.resolution)
print("pixel_size (y, x):", camera.pixel_size, "m")
print("adu_levels:", camera.adu_levels)
print("max pixel value:", camera.max_pixel_value)
print("exposure_bounds:", camera.exposure_bounds, "s")

# Both subclass the native Camera / SLM base classes.
print("camera is a native Camera:", isinstance(camera, Camera))
print("slm is a native SLM:", isinstance(slm, SLM))

## Exposing the camera and capturing an image

Here, we set the exposure by hand.



In [ ]:
camera.set_exposure(100e-6)
print(f"exposure now: {camera.get_exposure():.6f} s")

frame = camera.get_image()
print("frame shape (h, w):", frame.shape, "dtype:", frame.dtype)

image_grid(frame, "Camera image", cmap="turbo", colorbar_label="ADU").build()

The camera is underexposed. Autoexposing to a ``set_fraction`` of the full range.
Since the SLM currently displays a flat phase, only a focal spot is visible on the 
camera.



In [ ]:
camera.autoexpose(set_fraction=0.9)
print(f"exposure after autoexposure: {camera.get_exposure():.6f} s")

frame = camera.get_image()
image_grid(
    frame, "Camera image, autoexposed", cmap="turbo", colorbar_label="ADU"
).build()

## Regions of interest with ROI

``ROI`` is specified as ``(top_row, left_column, height, width)`` in camera pixels
``(row, col)``. We can zoom in on the focal spot by defining an ``ROI`` centered on
central camera pixel. By handing it to ``camera.set_roi``, ``camera.get_image`` then
returns only that window.



In [ ]:
center = (camera.resolution[0] // 2, camera.resolution[1] // 2)  # (row, col)
window = ROI.centered(center=center, size=(64, 64))
camera.set_roi(window)
print("current roi:", camera.roi)

cropped = camera.get_image()
print("cropped frame shape (h, w):", cropped.shape)

image_grid(cropped, "Region of interest", cmap="turbo", colorbar_label="ADU").build()

Passing ``None`` to ``camera.set_roi`` resets the camera to the full sensor.



In [ ]:
camera.set_roi(None)
print("roi after reset:", camera.roi)

## Normalize a device you already built

If you construct a device yourself, ``as_slm`` / ``as_camera`` return it ready to use.
This is exactly what ``open_slm`` / ``open_camera`` call internally. The simulated
devices implement the native interface directly, so they are passed through unchanged.
A real slmsuite driver is wrapped in an adapter here instead.



In [ ]:
raw_slm = SimulatedSLMTorch(input_geometry=slm_geometry, bitdepth=8)
print("raw device is a native SLM:", isinstance(raw_slm, SLM))

native = as_slm(raw_slm)
print("as_slm passes a native device through:", native is raw_slm)

# An already native device is returned unchanged. Calibrators rely on this to accept 
# either form.
print("camera as is:", as_camera(camera) is camera)

## Register a backend and open it by name

Name a driver once, then open it by that name anywhere. The symmetric
``register_camera_backend`` lets ``open_camera("mycam", ...)`` build a camera.



In [ ]:
register_slm_backend("simulated", SimulatedSLMTorch)
named_slm = open_slm("simulated", input_geometry=slm_geometry, bitdepth=8)
print("opened by name is a native SLM:", isinstance(named_slm, SLM))

## Connecting to real hardware
Nothing above is specific to simulation. With a real slmsuite driver you would
write one of the following, and every native call shown here works the same.

```python
from slmsuite.hardware.cameras.thorlabs import ThorCam

camera = open_camera(ThorCam, serial="12345")
```
or, if you already hold a driver instance:

```python
camera = as_camera(ThorCam(serial="12345"))
```
or register it once and open it by name:

```python
register_camera_backend("thorcam", ThorCam)
camera = open_camera("thorcam", serial="12345")
```
Every slmsuite driver already has a short name (thorlabs, basler, hamamatsu, ...).
Enable them all in one opt-in call, then open by name without importing the driver
yourself. Each vendor SDK is imported lazily, only when its backend is opened:

```python
from hologradpy.hardware import register_slmsuite_backends

register_slmsuite_backends()
camera = open_camera("thorlabs", serial="12345")
```


In [ ]:
plt.show()